# Create a batch deployment from a pipeline component stored in a registry

This example shows how to create a batch endpoint deployment from a pipeline component that is already registered in an Azure Machine Learning registry.

> Important: when you retrieve a pipeline component from a registry and use it in `PipelineComponentBatchDeployment`, pass the component **ID** (`component.id`) instead of the component object. This avoids SDK-side re-registration/validation issues for registry-backed pipeline components.


## 1. Connect to the workspace and registry

In this section, we connect to the workspace that will host the batch endpoint and to the registry that stores the pipeline component.


In [ ]:
%pip install azure-ai-ml==1.32.0

In [ ]:
from pathlib import Path

from azure.ai.ml import MLClient, load_component
from azure.ai.ml.entities import (
    AmlCompute,
    BatchEndpoint,
    PipelineComponentBatchDeployment,
)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential

Configure your workspace and registry details:


In [ ]:
subscription_id = "<SUBSCRIPTION_ID>"
resource_group = "<RESOURCE_GROUP>"
workspace = "<AML_WORKSPACE_NAME>"
registry_name = "<REGISTRY_NAME>"
component_name = "train_pipeline_component"
component_version = "1"

In [ ]:
credential = DefaultAzureCredential()
workspace_ml_client = MLClient(credential, subscription_id, resource_group, workspace)
registry_ml_client = MLClient(credential, registry_name=registry_name)

If you are running inside Azure Machine Learning and have a local config, you can connect to the workspace with `MLClient.from_config(DefaultAzureCredential())`.


## 2. Ensure compute exists

Batch deployments run on Azure Machine Learning compute attached to the workspace.


In [ ]:
compute_name = "batch-cluster"
if not any(
    filter(lambda m: m.name == compute_name, workspace_ml_client.compute.list())
):
    compute_cluster = AmlCompute(
        name=compute_name,
        description="Batch endpoints compute cluster",
        min_instances=0,
        max_instances=5,
    )
    workspace_ml_client.begin_create_or_update(compute_cluster).result()

## 3. Get the pipeline component from the registry

Retrieve the pipeline component from the registry.


In [ ]:
component_source = (
    Path.cwd().parents[5]
    / "cli/jobs/pipelines-with-components/pipeline_with_pipeline_component/pipeline_with_train_eval_pipeline_component/components/train_pipeline_component.yml"
)

try:
    pipeline_component = registry_ml_client.components.get(
        name=component_name, version=component_version
    )
except ResourceNotFoundError:
    pipeline_component = load_component(source=component_source)
    pipeline_component.name = component_name
    pipeline_component.version = component_version
    pipeline_component = registry_ml_client.components.create_or_update(
        pipeline_component
    )

print(pipeline_component.id)

## 4. Create the batch endpoint

Enable component deployments on the endpoint by setting `ComponentDeployment.Enabled` to `True`.


In [ ]:
import random
import string

endpoint_name = "registry-pipeline-batch"
endpoint_suffix = "".join(
    random.choice(string.ascii_lowercase + string.digits) for _ in range(5)
)
endpoint_name = f"{endpoint_name}-{endpoint_suffix}"
print(endpoint_name)

In [ ]:
endpoint = BatchEndpoint(
    name=endpoint_name,
    description="A batch endpoint backed by a pipeline component from registry",
    properties={"ComponentDeployment.Enabled": True},
)
workspace_ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

## 5. Create the deployment

Use the registry component **ID** when creating the deployment.


In [ ]:
deployment = PipelineComponentBatchDeployment(
    name="registry-pipeline-deployment",
    description="Batch deployment from a registry pipeline component",
    endpoint_name=endpoint_name,
    component=pipeline_component.id,
    settings={
        "continue_on_step_failure": False,
        "default_compute": compute_name,
    },
)
workspace_ml_client.batch_deployments.begin_create_or_update(deployment).result()

Once created, configure the deployment as the default deployment for the endpoint.


In [ ]:
endpoint = workspace_ml_client.batch_endpoints.get(endpoint_name)
endpoint.defaults.deployment_name = deployment.name
workspace_ml_client.batch_endpoints.begin_create_or_update(endpoint).result()